# SUS Hospitalization Rates - Recife Metropolitan Region

Which municipalities in the Recife Metropolitan Region (RMR) have the highest hospitalization rates in Brazil's public health system (SUS) per 1,000 inhabitants? Estimating this rate helps assess a population's access to hospitals and may signal how effective primary care is. It can work as a "thermometer" for health managers, helping to identify inequalities and to plan resources better.

#### Data and method

Data sources:
1. Hospitalizations: SIH/SUS (Hospital Information System of the Brazilian public health system), via [TabNet/DATASUS](http://tabnet.datasus.gov.br/cgi/deftohtm.exe?sih/cnv/nrpe.def), by place of residence, year of admission 2025
2. Population: [IBGE](https://www.ibge.gov.br/estatisticas/sociais/populacao/9103-estimativas-de-populacao.html) estimates (Brazilian Institute of Geography and Statistics), as of July 1, 2025
3. RMR definition: 14 municipalities, according to Pernambuco State Complementary Law 382/2018, as amended by [Complementary Law 426/2020](https://legis.alepe.pe.gov.br/texto.aspx?tiponorma=2&numero=426&complemento=0&ano=2020&tipo=&url=)

Method:
rate = hospitalizations ÷ population × 1,000.

In [1]:
import pandas as pd


In [2]:
internacoes_pe = pd.read_csv(
    'data/internacoes_pernambuco_atend_2024-2026_tabnet.csv', 
    sep = ';', encoding = 'latin-1', skiprows = 3, skipfooter = 6, engine = 'python'
)

In [3]:
internacoes_pe.shape

(186, 5)

In [4]:
internacoes_pe.dtypes

Município      str
2024         int64
2025         int64
2026         int64
Total        int64
dtype: object

#### 1. Loading the hospitalization data

The file exported from TabNet does not open directly in pandas. It needed four adjustments in `read_csv`:

- `encoding="latin-1"`: TabNet saves files in Latin-1, while pandas reads UTF-8 by default, which caused an error on accented characters
- `sep=";"`: columns are separated by `;`, while pandas expects `,` by default
- `skiprows=3`: skips the title lines at the top
- `skipfooter=6`: skips the footer lines with sources and notes

Check: the table has 186 rows and 5 columns (185 municipalities of Pernambuco + the Total row), with numbers read as int64.

In [5]:
internacoes_pe = internacoes_pe.rename(columns = { 'Município': 'municipio', '2025': 'internacoes_2025' })

In [6]:
internacoes_pe

,municipio,2024,internacoes_2025,2026,Total
0,260005 ABREU E LIMA,148,6566,4396,11110
1,260010 AFOGADOS DA INGAZEIRA,74,5310,3640,9024
2,260020 AFRANIO,41,1441,972,2454
3,260030 AGRESTINA,183,2348,1459,3990
4,260040 AGUA PRETA,37,1963,1269,3269
...,...,...,...,...,...
181,261620 VERTENTES,20,1156,823,1999
182,261630 VICENCIA,36,1837,1152,3025
183,261640 VITORIA DE SANTO ANTAO,251,10178,6484,16913
184,261650 XEXEU,17,1092,657,1766


In [7]:
internacoes_pe = internacoes_pe.drop(columns= ['2024', '2026', 'Total'])

In [8]:
internacoes_pe

,municipio,internacoes_2025
0,260005 ABREU E LIMA,6566
1,260010 AFOGADOS DA INGAZEIRA,5310
2,260020 AFRANIO,1441
3,260030 AGRESTINA,2348
4,260040 AGUA PRETA,1963
...,...,...
181,261620 VERTENTES,1156
182,261630 VICENCIA,1837
183,261640 VITORIA DE SANTO ANTAO,10178
184,261650 XEXEU,1092


In [9]:
internacoes_pe["codigo_municipio"] = internacoes_pe["municipio"].str[:6]

In [10]:
internacoes_pe

,municipio,internacoes_2025,codigo_municipio
0,260005 ABREU E LIMA,6566,260005
1,260010 AFOGADOS DA INGAZEIRA,5310,260010
2,260020 AFRANIO,1441,260020
3,260030 AGRESTINA,2348,260030
4,260040 AGUA PRETA,1963,260040
...,...,...,...
181,261620 VERTENTES,1156,261620
182,261630 VICENCIA,1837,261630
183,261640 VITORIA DE SANTO ANTAO,10178,261640
184,261650 XEXEU,1092,261650


In [11]:
internacoes_pe["municipio"] = internacoes_pe["municipio"].str[7:]

In [12]:
internacoes_pe

,municipio,internacoes_2025,codigo_municipio
0,ABREU E LIMA,6566,260005
1,AFOGADOS DA INGAZEIRA,5310,260010
2,AFRANIO,1441,260020
3,AGRESTINA,2348,260030
4,AGUA PRETA,1963,260040
...,...,...,...
181,VERTENTES,1156,261620
182,VICENCIA,1837,261630
183,VITORIA DE SANTO ANTAO,10178,261640
184,XEXEU,1092,261650


In [13]:
internacoes_pe = internacoes_pe.loc[internacoes_pe["codigo_municipio"] != "Total"]

In [14]:
internacoes_pe

,municipio,internacoes_2025,codigo_municipio
0,ABREU E LIMA,6566,260005
1,AFOGADOS DA INGAZEIRA,5310,260010
2,AFRANIO,1441,260020
3,AGRESTINA,2348,260030
4,AGUA PRETA,1963,260040
...,...,...,...
180,VERTENTE DO LERIO,433,261618
181,VERTENTES,1156,261620
182,VICENCIA,1837,261630
183,VITORIA DE SANTO ANTAO,10178,261640


In [15]:
internacoes_pe.isnull().sum()

municipio           0
internacoes_2025    0
codigo_municipio    0
dtype: int64

#### 2. Cleaning

- Columns kept: only `2025`, renamed to `internacoes_2025` (hospitalizations in 2025). Column `2024` only holds 2024 hospitalizations that were processed late, `2026` is an incomplete year (still subject to updates), and `Total` adds up different years, so all three were dropped.
- Code and name split: TabNet stores the municipality as `"260005 ABREU E LIMA"`. The first 6 characters became `codigo_municipio` (municipality code), and the name stayed in `municipio` (from position 7 on, to leave out the space).
- The "Total" row was removed, since it is not a municipality.
- Dashes `-`: TabNet's legend uses `-` for zero, but this file has none. Numbers were already read as integers.
- Missing values: checked with `isnull().sum()`, none found.

Check: 185 municipalities of Pernambuco, 3 columns.

In [16]:
codigos_rmr = ["260005", "260105", "260290", "260345", "260680", "260760", "260720", "260775", "260790","260940", "260960", "261070", "261160", "261370"]

In [17]:
internacoes_rmr = internacoes_pe.loc[internacoes_pe["codigo_municipio"].isin(codigos_rmr)]

In [18]:
internacoes_rmr

,municipio,internacoes_2025,codigo_municipio
0,ABREU E LIMA,6566,260005
11,ARACOIABA,1218,260105
30,CABO DE SANTO AGOSTINHO,14698,260290
36,CAMARAGIBE,12093,260345
74,IGARASSU,8595,260680
78,IPOJUCA,7474,260720
82,ILHA DE ITAMARACA,1480,260760
85,ITAPISSUMA,1891,260775
87,JABOATAO DOS GUARARAPES,49138,260790
108,MORENO,3838,260940


#### 3. Filtering the RMR

The RMR has 14 municipalities, according to Complementary Law 382/2018 as amended by Complementary Law 426/2020, which removed Goiana. The [PDUI-RMR](https://www.pdui-rmr.pe.gov.br/municipios) website still lists 15, but it is outdated.

The filter uses the **municipality code** instead of the name, because the code is unique and does not change with accents, capitalization or spacing.

Check: 14 rows.

In [19]:
pd.ExcelFile("data/ibge_populacao_2025.xls").sheet_names

C:\Users\lalap\AppData\Local\Temp\ipykernel_14016\151845170.py:1: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  pd.ExcelFile("data/ibge_populacao_2025.xls").sheet_names


['BRASIL E UFs', 'Municípios']

In [20]:
municipios = pd.read_excel("data/ibge_populacao_2025.xls", sheet_name = "Municípios", skiprows = 1, dtype={"COD. UF": str, "COD. MUNIC": str} )

C:\Users\lalap\AppData\Local\Temp\ipykernel_14016\627259760.py:1: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  municipios = pd.read_excel("data/ibge_populacao_2025.xls", sheet_name = "Municípios", skiprows = 1, dtype={"COD. UF": str, "COD. MUNIC": str} )


In [21]:
municipios

,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,POPULAÇÃO ESTIMADA,Unnamed: 5
0,RO,11,00015,Alta Floresta D'Oeste,22787.0,NaN
1,RO,11,00023,Ariquemes,109170.0,NaN
2,RO,11,00031,Cabixi,5664.0,NaN
3,RO,11,00049,Cacoal,98280.0,NaN
4,RO,11,00056,Cerejeiras,16966.0,NaN
...,...,...,...,...,...,...
5596,22,População judicial do município de Urucará-AM:...,NaN,NaN,NaN,NaN
5597,23,População judicial do município de Urucurituba...,NaN,NaN,NaN,NaN
5598,24,População judicial do município de Rosário-MA:...,NaN,NaN,NaN,NaN
5599,25,População judicial do município de Sao Domingo...,NaN,NaN,NaN,NaN


In [22]:
municipios = municipios.loc[municipios["COD. UF"] == "26"]

In [23]:
municipios


,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,POPULAÇÃO ESTIMADA,Unnamed: 5
1465,PE,26,00054,Abreu e Lima,104248.0,NaN
1466,PE,26,00104,Afogados da Ingazeira,42672.0,NaN
1467,PE,26,00203,Afrânio,19409.0,NaN
1468,PE,26,00302,Agrestina,24680.0,NaN
1469,PE,26,00401,Água Preta,26981.0,NaN
...,...,...,...,...,...,...
1645,PE,26,16183,Vertente do Lério,7776.0,NaN
1646,PE,26,16209,Vertentes,23127.0,NaN
1647,PE,26,16308,Vicência,27151.0,NaN
1648,PE,26,16407,Vitória de Santo Antão,144243.0,NaN


In [24]:
municipios["codigo_municipio"] = (municipios["COD. UF"] + municipios["COD. MUNIC"]).str[:6]

In [25]:
municipios

,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,POPULAÇÃO ESTIMADA,Unnamed: 5,codigo_municipio
1465,PE,26,00054,Abreu e Lima,104248.0,NaN,260005
1466,PE,26,00104,Afogados da Ingazeira,42672.0,NaN,260010
1467,PE,26,00203,Afrânio,19409.0,NaN,260020
1468,PE,26,00302,Agrestina,24680.0,NaN,260030
1469,PE,26,00401,Água Preta,26981.0,NaN,260040
...,...,...,...,...,...,...,...
1645,PE,26,16183,Vertente do Lério,7776.0,NaN,261618
1646,PE,26,16209,Vertentes,23127.0,NaN,261620
1647,PE,26,16308,Vicência,27151.0,NaN,261630
1648,PE,26,16407,Vitória de Santo Antão,144243.0,NaN,261640


In [26]:
municipios = municipios.drop(columns= ["Unnamed: 5", "NOME DO MUNICÍPIO", "COD. UF", "COD. MUNIC", "UF"])
municipios 

,POPULAÇÃO ESTIMADA,codigo_municipio
1465,104248.0,260005
1466,42672.0,260010
1467,19409.0,260020
1468,24680.0,260030
1469,26981.0,260040
...,...,...
1645,7776.0,261618
1646,23127.0,261620
1647,27151.0,261630
1648,144243.0,261640


In [27]:
municipios = municipios.rename(columns = {"POPULAÇÃO ESTIMADA" : "populacao_2025"})

In [28]:
municipios.head(5)
municipios.dtypes

populacao_2025      float64
codigo_municipio        str
dtype: object

In [29]:
municipios["populacao_2025"] = municipios["populacao_2025"].astype("int64")

In [30]:
municipios.head(5)
municipios.dtypes

populacao_2025      int64
codigo_municipio      str
dtype: object

In [31]:
municipios = municipios.set_index("codigo_municipio")
municipios 

,populacao_2025
codigo_municipio,
260005,104248
260010,42672
260020,19409
260030,24680
260040,26981
...,...
261618,7776
261620,23127
261630,27151


In [32]:
internacoes_rmr = internacoes_rmr.set_index("codigo_municipio")
internacoes_rmr

,municipio,internacoes_2025
codigo_municipio,,
260005,ABREU E LIMA,6566
260105,ARACOIABA,1218
260290,CABO DE SANTO AGOSTINHO,14698
260345,CAMARAGIBE,12093
260680,IGARASSU,8595
260720,IPOJUCA,7474
260760,ILHA DE ITAMARACA,1480
260775,ITAPISSUMA,1891
260790,JABOATAO DOS GUARARAPES,49138


In [33]:
rmr = internacoes_rmr.join(municipios)
rmr


,municipio,internacoes_2025,populacao_2025
codigo_municipio,,,
260005,ABREU E LIMA,6566,104248
260105,ARACOIABA,1218,19996
260290,CABO DE SANTO AGOSTINHO,14698,218049
260345,CAMARAGIBE,12093,156112
260680,IGARASSU,8595,123017
260720,IPOJUCA,7474,106539
260760,ILHA DE ITAMARACA,1480,25658
260775,ITAPISSUMA,1891,29669
260790,JABOATAO DOS GUARARAPES,49138,684293


In [34]:

rmr["taxa_por_mil_hab"] = rmr["internacoes_2025"] / rmr["populacao_2025"] * 1000
rmr


,municipio,internacoes_2025,populacao_2025,taxa_por_mil_hab
codigo_municipio,,,,
260005,ABREU E LIMA,6566,104248,62.984422
260105,ARACOIABA,1218,19996,60.912182
260290,CABO DE SANTO AGOSTINHO,14698,218049,67.406867
260345,CAMARAGIBE,12093,156112,77.463616
260680,IGARASSU,8595,123017,69.868392
260720,IPOJUCA,7474,106539,70.152714
260760,ILHA DE ITAMARACA,1480,25658,57.681815
260775,ITAPISSUMA,1891,29669,63.736560
260790,JABOATAO DOS GUARARAPES,49138,684293,71.808421


#### 4. Population and rate

- `Municípios` sheet of the IBGE file (the first sheet only has states), skipping 1 title line.
- `dtype=str` on the code columns, so the leading zeros are kept (`00015` was turning into `15.0`).
- IBGE uses 7-digit codes (the last one is a check digit) and DATASUS uses 6, so I joined `COD. UF` and `COD. MUNIC` and kept the first 6 digits.
- Filtered Pernambuco (state code 26): 185 municipalities, the same number as in TabNet.
- `join` on the code: all 14 RMR municipalities with population filled in.

Manual check: Recife = 124,014 ÷ 1,588,376 × 1,000 = 78.08.

In [35]:
rmr = rmr.sort_values(by = "taxa_por_mil_hab", ascending = False)
rmr

,municipio,internacoes_2025,populacao_2025,taxa_por_mil_hab
codigo_municipio,,,,
260960,OLINDA,30016,364717,82.299427
261370,SAO LOURENCO DA MATA,9397,118258,79.461855
261160,RECIFE,124014,1588376,78.075972
260345,CAMARAGIBE,12093,156112,77.463616
260790,JABOATAO DOS GUARARAPES,49138,684293,71.808421
260720,IPOJUCA,7474,106539,70.152714
260680,IGARASSU,8595,123017,69.868392
260290,CABO DE SANTO AGOSTINHO,14698,218049,67.406867
260940,MORENO,3838,57654,66.569536


In [36]:
rmr.to_csv("data/taxa_internacao_rmr_2025.csv")


#### 5. Result

Table sorted from the highest to the lowest rate, saved to `data/taxa_internacao_rmr_2025.csv` with the municipality code as index, for the map in Power BI.

#### 6. Conclusions

Looking at the hospitalization rate per 1,000 inhabitants, Olinda (82.3), São Lourenço da Mata (79.5) and Recife (78.1) lead the RMR. Paulista (58.4) and Ilha de Itamaracá (57.7) have the lowest rates: Olinda's rate is about 43% higher than Itamaracá's.

Even though Recife has by far the largest number of hospitalizations, it is not at the top of the list, because the rate takes each municipality's population size into account.

**Health interpretation**

A high hospitalization rate can point to several issues, such as less effective primary care, more illness in the population, or an older population (older people are hospitalized more often). It may also reflect lower private health insurance coverage: people with insurance usually go to private hospitals, which are not included in the SIH.

On the other hand, a low rate is not always good news: it may indicate poorer access to hospitals, as in municipalities farther from Recife, where most hospital beds are.

This analysis shows **where** to look, but it does not prove the causes. The next steps would be an age-standardized rate and the rate of hospitalizations for ambulatory care-sensitive conditions (ICSAP, in Portuguese).

**Limitations**

- **SUS only:** the analysis covers only public health system hospitalizations. People hospitalized in private hospitals are left out, so the rate does not represent all hospitalizations of the population.
- **Hospitalizations, not people:** each hospitalization is counted separately. If the same person was hospitalized more than once in the year, they count more than once. Municipalities with many readmissions get higher rates.
- **Address on the AIH:** each hospitalization is assigned to the municipality recorded on the hospitalization form (AIH). Outdated addresses, or addresses different from the actual residence, may assign a hospitalization to the wrong municipality.
- **Crude rate:** the rate is not broken down by age or by cause of hospitalization. Municipalities with more older residents, for example, tend to have higher rates without this meaning worse care.
- **Data subject to updates:** the SIH can still receive late AIHs, so the 2025 numbers may still change slightly.

**Sources**

- Hospitalizations: Brazilian Ministry of Health, SIH/SUS, via [TabNet/DATASUS](http://tabnet.datasus.gov.br/cgi/deftohtm.exe?sih/cnv/nrpe.def). Accessed in November 2026.
- Population: [IBGE, Population Estimates](https://www.ibge.gov.br/estatisticas/sociais/populacao/9103-estimativas-de-populacao.html), as of July 1, 2025.
- RMR composition: Lei Complementar Estadual No. 382/2018, as amended by [Complementary Law No. 426/2020](https://legis.alepe.pe.gov.br/texto.aspx?tiponorma=2&numero=426&complemento=0&ano=2020&tipo=&url=).